In [3]:
# Cell 1
# Mục đích: tạo kết nối BigQuery và các hàm dùng chung (đo byte, chạy query có trần chi phí).
import os
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv()                                   # đọc file .env vào biến môi trường
PROJECT = os.getenv("GCP_PROJECT", "")          # "" là giá trị mặc định nếu thiếu biến
LOCATION = os.getenv("BQ_LOCATION", "US")
client = bigquery.Client(project=PROJECT, location=LOCATION)   # kênh kết nối tới BigQuery

def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi đọc quá sớm lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng byte, không chạy thật, không tốn tiền
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Sẽ quét: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật; trần là maximum_bytes_billed (đổi MB sang byte bằng * 1024 * 1024)
    cfg = bigquery.QueryJobConfig(use_query_cache=False,
                                  maximum_bytes_billed=max_mb * 1024 * 1024)
    job = client.query(sql, job_config=cfg)          # gửi job, không chờ
    df = job.result().to_dataframe()                 # chờ xong rồi đưa kết quả vào df
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

d:\Installsoftware\Anaconda\envs\bi\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.16) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
d:\Installsoftware\Anaconda\envs\bi\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.16) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)
d:\Installsoftware\Anaconda\envs\bi\Lib\site-packages\google\api_cor

In [ ]:
# Cell 2
# Mục đích: so số NULL trước và sau SAFE_CAST trên bảng raw. Nếu hai số bằng nhau
# thì ép kiểu không làm hỏng ngày nào (NULL sẵn có ở raw là bình thường, ví dụ đơn chưa giao).
sql = f"""
select
    countif(order_purchase_timestamp is null)                                  as raw_null_purchased,
    countif(safe_cast(order_purchase_timestamp as timestamp) is null)          as cast_null_purchased,
    countif(order_approved_at is null)                                         as raw_null_approved,
    countif(safe_cast(order_approved_at as timestamp) is null)                 as cast_null_approved,
    countif(order_delivered_carrier_date is null)                              as raw_null_carrier,
    countif(safe_cast(order_delivered_carrier_date as timestamp) is null)      as cast_null_carrier,
    countif(order_delivered_customer_date is null)                             as raw_null_customer,
    countif(safe_cast(order_delivered_customer_date as timestamp) is null)     as cast_null_customer,
    countif(order_estimated_delivery_date is null)                             as raw_null_estimated,
    countif(safe_cast(order_estimated_delivery_date as timestamp) is null)     as cast_null_estimated
from `{PROJECT}.olist_raw.orders`
"""
check(sql)
df = stats(sql)
df.T      # xoay thành 10 dòng x 1 cột cho dễ đọc

Sẽ quét: 9.86 MB
processed: 9.86 MB | billed: 10.0 MB


,0
raw_null_purchased,0
cast_null_purchased,0
raw_null_approved,160
cast_null_approved,160
raw_null_carrier,1783
cast_null_carrier,1783
raw_null_customer,2965
cast_null_customer,2965
raw_null_estimated,0
cast_null_estimated,0


In [ ]:
# Cell 3
# Mục đích: kiểm tra view staging: số dòng, khóa duy nhất, khoảng ngày (khớp Ngày 2: 2016-09-04 đến 2018-10-17).
sql = f"""
select
    count(*)                       as so_dong,
    count(distinct order_id)       as so_id,
    min(date(purchased_at))        as ngay_dau,
    max(date(purchased_at))        as ngay_cuoi,
    countif(purchased_at is null)  as null_purchased
from `{PROJECT}.olist_dbt_dev.stg_olist__orders`
"""
df = stats(sql)
df.T

processed: 5.22 MB | billed: 10.0 MB


,0
so_dong,99441
so_id,99441
ngay_dau,2016-09-04
ngay_cuoi,2018-10-17
null_purchased,0


In [ ]:
# Cell 4
# Mục đích: xem danh sách order_status đầy đủ (câu còn mở từ Ngày 2, output lần đó bị cắt).
sql = f"""
select order_status, count(*) as so_dong
from `{PROJECT}.olist_dbt_dev.stg_olist__orders`
group by order_status
order by so_dong desc
"""
df = stats(sql)
print(len(df))
df

processed: 1.04 MB | billed: 10.0 MB
8


,order_status,so_dong
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2
